In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [34]:
!pip install -q openai psycopg2-binary tiktoken langchain-text-splitters \
    python-dotenv marker-pdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 110.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.7/195.7 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.2/223.2 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 126.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.4/226.4 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2

<frozen posixpath>:82: RuntimeWarning: coroutine 'get_links_async' was never awaited


In [ ]:
env_content = """
SUPABASE_URL=your_supabase_url_here
OPENAI_API_KEY=your_openai_api_key_here
"""

with open("/content/drive/MyDrive/rag_project/.env", "w") as f:
    f.write(env_content)

In [ ]:
!ls -a /content/drive/MyDrive/rag_project

data  .env  ingestion


In [36]:
import sys
import os

PROJECT_ROOT = "/content/drive/MyDrive/rag_project"
sys.path.insert(0, PROJECT_ROOT)

# Load .env
from dotenv import load_dotenv
load_dotenv(f"{PROJECT_ROOT}/.env")

# Verify env vars loaded
print("SUPABASE_URL set:", bool(os.getenv("SUPABASE_URL")))
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))

SUPABASE_URL set: True
OPENAI_API_KEY set: True


In [ ]:
from ingestion.ingest_folder import ingest_folder

PDF_FOLDER = f"{PROJECT_ROOT}/data"
ingest_folder(PDF_FOLDER)

Starting ingestion of 3 files...

Ingesting: B.Tech_IT_Curriculam_Syllabus_2022-2023.pdf
Document ID: 58801612-2c91-4f27-b932-18a8fb30a56d
✅ document_id confirmed: 58801612-2c91-4f27-b932-18a8fb30a56d


Recognizing Text: 100%|██████████| 12260/12260 [04:52<00:00, 41.96it/s]


12 chunks are <100 chars
Ingested 480 chunks in 2 batches
✅ Success: B.Tech_IT_Curriculam_Syllabus_2022-2023.pdf (480 chunks)
Ingesting: BCI-2024-2025.pdf
Document ID: b2304afd-70d6-475e-9dfc-b079e229f5b5
✅ document_id confirmed: b2304afd-70d6-475e-9dfc-b079e229f5b5


Recognizing Text: 100%|██████████| 6164/6164 [02:01<00:00, 50.55it/s] 


1 chunks are <100 chars
Ingested 161 chunks in 1 batches
✅ Success: BCI-2024-2025.pdf (161 chunks)
Ingesting: BKT-2024-2025.pdf
Document ID: 0a5ab412-029d-4868-bf65-b2f5c257af8f
✅ document_id confirmed: 0a5ab412-029d-4868-bf65-b2f5c257af8f


Recognizing Text: 100%|██████████| 6061/6061 [02:34<00:00, 39.24it/s] 


1 chunks are <100 chars
Ingested 164 chunks in 1 batches
✅ Success: BKT-2024-2025.pdf (164 chunks)

Ingestion completed.


In [29]:
# Add to Cell 1
!pip install -q requests beautifulsoup4 playwright nest_asyncio
!playwright install chromium

/usr/lib/python3.12/pathlib.py:407: RuntimeWarning: coroutine 'get_links_async' was never awaited
  def _load_parts(self):


In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [30]:
# Cell 3 — paste this entire cell, replacing everything you have

import os, re, time, hashlib, requests
import nest_asyncio
nest_asyncio.apply()
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup
import pandas as pd
from IPython.display import display
import asyncio
from playwright.async_api import async_playwright

OUTPUT_DIR = "/content/drive/MyDrive/rag_project/data"
DELAY      = 1.5
TIMEOUT    = 15
MAX_PDF_MB = 50
HEADERS    = {"User-Agent": "Mozilla/5.0 (compatible; VIT-Research-Bot/1.0)"}

SEED_URLS = [
    "https://vit.ac.in/academics",
    "https://vit.ac.in/admissions",
    "https://vit.ac.in/research",
    "https://vit.ac.in/campus-life/hostels",
    "https://vit.ac.in/placements",
    "https://vit.ac.in/schools/sense",
    "https://vit.ac.in/schools/scope",
    "https://vit.ac.in/schools/site",
    "https://vit.ac.in/schools/select",
    "https://vit.ac.in/schools/smec",
    "https://vit.ac.in/schools/sbst",
    "https://vit.ac.in/schools/ssl",
    "https://vit.ac.in/schools/design",
    "https://vit.ac.in/schools/sense/academics",
    "https://vit.ac.in/schools/scope/academics",
    "https://vit.ac.in/schools/site/academics",
]

def is_vit_domain(url):
    return "vit.ac.in" in urlparse(url).netloc.lower()

def url_to_filename(url):
    path = urlparse(url).path
    name = os.path.basename(path)
    if not name.lower().endswith(".pdf"):
        name = hashlib.md5(url.encode()).hexdigest()[:12] + ".pdf"
    return re.sub(r"[^\w\-.]", "_", name)

def get_pdf_size(url):
    try:
        r = requests.head(url, headers=HEADERS, timeout=8, allow_redirects=True)
        cl = r.headers.get("Content-Length")
        if cl:
            kb = int(cl) / 1024
            return f"{kb:.0f} KB" if kb < 1024 else f"{kb/1024:.1f} MB"
    except:
        pass
    return "unknown"

async def get_links_async(url):
    try:
        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=True)
            page = await browser.new_page()
            await page.goto(url, timeout=20000, wait_until="networkidle")
            content = await page.content()
            await browser.close()
        soup = BeautifulSoup(content, "html.parser")
        pdfs, subpages = [], []
        for tag in soup.find_all("a", href=True):
            href = tag["href"].strip()
            if not href or href.startswith("javascript"):
                continue
            full = urljoin(url, href)
            if full.lower().endswith(".pdf") and is_vit_domain(full):
                pdfs.append(full)
            elif (is_vit_domain(full)
                  and urlparse(full).scheme in ("http", "https")
                  and "#" not in full and full != url):
                subpages.append(full)
        return list(set(pdfs)), list(set(subpages))
    except Exception as e:
        print(f"  ⚠️  {url}: {e}")
        return [], []

def get_links(url):
    # Colab uses an existing event loop — can't use run_until_complete
    return asyncio.get_event_loop().run_until_complete(get_links_async(url))

print("✅ Config ready. OUTPUT_DIR =", OUTPUT_DIR)

✅ Config ready. OUTPUT_DIR = /content/drive/MyDrive/rag_project/data


In [31]:
# Cell 4 — preview

def preview_pdfs(seed_urls, max_pages=120):
    visited, queue = set(), list(seed_urls)
    found = {}

    while queue and len(visited) < max_pages:
        url = queue.pop(0)
        if url in visited:
            continue
        visited.add(url)
        print(f"🔍 [{len(visited)}/{max_pages}] {url}")

        pdfs, subpages = get_links(url)

        for pdf_url in pdfs:
            if pdf_url not in found:
                found[pdf_url] = {
                    "filename": url_to_filename(pdf_url),
                    "source_page": url,
                    "url": pdf_url,
                }
                print(f"  📄 {url_to_filename(pdf_url)}")

        if url in seed_urls:
            for link in subpages:
                if link not in visited:
                    queue.append(link)

        time.sleep(DELAY)

    print(f"\n{'='*60}")
    print(f"Pages crawled: {len(visited)} | PDFs found: {len(found)}")
    print("Checking sizes...")

    rows, known_kb = [], []
    for url, meta in found.items():
        size = get_pdf_size(url)
        rows.append({"filename": meta["filename"], "size": size,
                     "source_page": meta["source_page"], "url": url})
        if "KB" in size: known_kb.append(float(size.replace(" KB","")))
        elif "MB" in size: known_kb.append(float(size.replace(" MB",""))*1024)

    if known_kb:
        print(f"Estimated total: ~{sum(known_kb)/1024:.1f} MB\n")

    df = pd.DataFrame(rows)[["filename","size","source_page","url"]] if rows else pd.DataFrame()
    display(df)
    return found

found_pdfs = preview_pdfs(SEED_URLS)

🔍 [1/120] https://vit.ac.in/academics
🔍 [2/120] https://vit.ac.in/admissions
  📄 Academic-Regulations.pdf
  📄 Student-Code-of-Conduct.pdf
  📄 Awarness-Circular.pdf
  📄 Instructions-to-the-selected-candidates-VITREE-January-2026.pdf
  📄 VITREE-2026-Hall_Ticket_Download_Steps.pdf
  📄 e-SANAD-Notification.pdf
🔍 [3/120] https://vit.ac.in/research
🔍 [4/120] https://vit.ac.in/campus-life/hostels
🔍 [5/120] https://vit.ac.in/placements
🔍 [6/120] https://vit.ac.in/schools/sense
  📄 SENSE-HACK.pdf
🔍 [7/120] https://vit.ac.in/schools/scope
🔍 [8/120] https://vit.ac.in/schools/site
🔍 [9/120] https://vit.ac.in/schools/select
🔍 [10/120] https://vit.ac.in/schools/smec
  📄 B.Tech-Mechanical-Engineering-Placement.pdf
  📄 SMEC-Type-of-innovation.pdf
🔍 [11/120] https://vit.ac.in/schools/sbst
  📄 Declaration-best-innovation.pdf
  📄 SBST-Presentation-2024-2025-updated.pdf
  📄 DHR-ICMR-Health-Research-Excellence-Summit-2024.pdf
  📄 SBST-testing-samples-consultancy.pdf
🔍 [12/120] https://vit.ac.in/schools/ssl

,filename,size,source_page,url
0,Academic-Regulations.pdf,1.4 MB,https://vit.ac.in/admissions,https://vit.ac.in/sites/default/files/academic...
1,Student-Code-of-Conduct.pdf,739 KB,https://vit.ac.in/admissions,https://vit.ac.in/wp-content/uploads/2023/11/S...
2,Awarness-Circular.pdf,353 KB,https://vit.ac.in/admissions,https://vit.ac.in/files/prospectus/Awarness-Ci...
3,Instructions-to-the-selected-candidates-VITREE...,476 KB,https://vit.ac.in/admissions,https://vit.ac.in/files/Instructions-to-the-se...
4,VITREE-2026-Hall_Ticket_Download_Steps.pdf,949 KB,https://vit.ac.in/admissions,https://vit.ac.in/files/prospectus/VITREE-2026...
...,...,...,...,...
148,Int.M.Sc.2026-2027-About-the-Programme.pdf,1.9 MB,https://admissions.vit.ac.in/integrated-msc-20...,https://vit.ac.in/files/prospectus/Int.M.Sc.20...
149,Int-M.Sc-Information-Brochure-2026.pdf,38.8 MB,https://admissions.vit.ac.in/integrated-msc-20...,https://vit.ac.in/files/prospectus/Int-M.Sc-In...
150,Anti-Ragging-Team-2025-2026.pdf,523 KB,https://vit.ac.in/anti-ragging-committee,https://vit.ac.in/wp-content/uploads/2025/09/A...
151,M.Sc.Programme-Without-Entrance-Information-Br...,55.2 MB,https://admissions.vit.ac.in/msc-2026-applicat...,https://vit.ac.in/files/prospectus/M.Sc.Progra...


In [33]:
# Cell 5 — Download

def download_all(found, skip_keywords=None):
    skip_keywords = skip_keywords or []
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    existing = set(os.listdir(OUTPUT_DIR))

    total = len(found)
    downloaded = skipped = failed = 0

    for i, (url, meta) in enumerate(found.items(), 1):
        filename = meta["filename"]

        if any(kw.lower() in filename.lower() for kw in skip_keywords):
            print(f"[{i}/{total}] ⏭️  Filtered: {filename}")
            skipped += 1
            continue

        dest = os.path.join(OUTPUT_DIR, filename)
        if filename in existing or (os.path.exists(dest) and os.path.getsize(dest) > 1024):
            print(f"[{i}/{total}] ⏭️  Already have: {filename}")
            skipped += 1
            continue

        try:
            r = requests.get(url, headers=HEADERS, timeout=TIMEOUT, stream=True)
            r.raise_for_status()
            cl = int(r.headers.get("Content-Length", 0))
            if cl > MAX_PDF_MB * 1024 * 1024:
                print(f"[{i}/{total}] ⏭️  Too large ({cl//1024//1024}MB): {filename}")
                skipped += 1
                continue
            with open(dest, "wb") as f:
                for chunk in r.iter_content(8192):
                    f.write(chunk)
            size_kb = os.path.getsize(dest) // 1024
            print(f"[{i}/{total}] ✅ {filename} ({size_kb} KB)")
            downloaded += 1
        except Exception as e:
            print(f"[{i}/{total}] ❌ {filename}: {e}")
            failed += 1
            if os.path.exists(dest):
                os.remove(dest)
        time.sleep(0.3)

    total_bytes = sum(
        os.path.getsize(os.path.join(OUTPUT_DIR, f))
        for f in os.listdir(OUTPUT_DIR)
        if os.path.isfile(os.path.join(OUTPUT_DIR, f))
    )
    print(f"\n{'='*60}")
    print(f"Downloaded: {downloaded} | Skipped: {skipped} | Failed: {failed}")
    print(f"Folder size: {total_bytes/1024/1024:.1f} MB")

download_all(found_pdfs, skip_keywords=[
    "Minutes-of-the",
    "Phy-Edu-Achievements", "Sports-Achievements", "EventAchievements", "sports_achievements",
    "Events-20", "Events-AY",
    "Newletter", "Newsletter",
    "ssl-conference", "SSL-International", "IEA-conference", "SSl-Conference",
    "SENSE-HACK", "Engineers-day", "DHR-ICMR",
    "Affidavit", "SponsorshipLetter", "EmployerCertificate", "Undertaking", "PhysicalFitness",
    "Country-Codes", "SMEC-Type", "Declaration-best",
    "Acad-Calen-for-Trimester",
    "Acad-Calen-for-Fall-Semester-2022", "Acad-Calen-for-Fall-Semester-2023",
    "Acad-Calen-for-Winter-Semester-2022", "Acad-Calen-for-Winter-Semester-2023",
    "VITREE-January-2025",
    "selected-candidates-instructions-july-2024",
    "Personal-Interview-Details",
    "VITREE-2026-Hall_Ticket",
    "Instructions-to-the-selected",
    "Awarness-Circular",
    "template-for-two-page", "Template-Deep-Tech",
])

[1/153] ✅ Academic-Regulations.pdf (1425 KB)
[2/153] ✅ Student-Code-of-Conduct.pdf (738 KB)
[3/153] ⏭️  Filtered: Awarness-Circular.pdf
[4/153] ⏭️  Filtered: Instructions-to-the-selected-candidates-VITREE-January-2026.pdf
[5/153] ⏭️  Filtered: VITREE-2026-Hall_Ticket_Download_Steps.pdf
[6/153] ✅ e-SANAD-Notification.pdf (530 KB)
[7/153] ⏭️  Filtered: SENSE-HACK.pdf
[8/153] ✅ B.Tech-Mechanical-Engineering-Placement.pdf (432 KB)
[9/153] ⏭️  Filtered: SMEC-Type-of-innovation.pdf
[10/153] ⏭️  Filtered: Declaration-best-innovation.pdf
[11/153] ✅ SBST-Presentation-2024-2025-updated.pdf (9725 KB)
[12/153] ⏭️  Filtered: DHR-ICMR-Health-Research-Excellence-Summit-2024.pdf
[13/153] ✅ SBST-testing-samples-consultancy.pdf (333 KB)
[14/153] ⏭️  Filtered: ssl-conference-july-2020.pdf
[15/153] ⏭️  Filtered: SSL-International-Conference1.pdf
[16/153] ⏭️  Filtered: IEA-conference-Brouchure.pdf
[17/153] ⏭️  Filtered: SSl-Conference-Deatails.pdf
[18/153] ⏭️  Filtered: ssl-conference-aug-2020.pdf
[19/153]

In [38]:
from ingestion.ingest_folder import ingest_folder
ingest_folder("/content/drive/MyDrive/rag_project/data")

Starting ingestion of 59 PDFs with 3 workers...

Finished: Acad-Calen-Revised-for-Winter-Semester-2022-23-for-Seniors-15-11-2022.pdf
❌ Failed: Acad-Calen-Revised-for-Winter-Semester-2022-23-for-Seniors-15-11-2022.pdf — OperationalError: SSL SYSCALL error: EOF detected

Traceback (most recent call last):
  File "/content/drive/MyDrive/rag_project/ingestion/ingest_folder.py", line 31, in ingest_single
  File "/content/drive/MyDrive/rag_project/ingestion/final_ingestion.py", line 158, in document_exists_by_fingerprint
    cursor.execute(
psycopg2.OperationalError: SSL SYSCALL error: EOF detected


Finished: Acad-Calen-for-MBA-Freshers-2023-24.pdf
❌ Failed: Acad-Calen-for-MBA-Freshers-2023-24.pdf — OperationalError: SSL SYSCALL error: EOF detected

Traceback (most recent call last):
  File "/content/drive/MyDrive/rag_project/ingestion/ingest_folder.py", line 31, in ingest_single
  File "/content/drive/MyDrive/rag_project/ingestion/final_ingestion.py", line 158, in document_exists_by_finger

In [39]:
# Run this cell to overwrite ingest_folder.py

content = '''import os
import uuid
import traceback

from ingestion.scan import extract_text_from_pdf
from ingestion.chunking import chunk_markdown_document
from ingestion.final_ingestion import (
    embed_and_insert,
    insert_document,
    compute_fingerprint,
    update_document_status,
    document_exists_by_fingerprint
)


def ingest_single(pdf_path: str, filename: str) -> str:
    document_name = os.path.splitext(filename)[0]
    document_id = str(uuid.uuid4())

    try:
        fingerprint = compute_fingerprint(pdf_path)
        if document_exists_by_fingerprint(fingerprint):
            return f"\\u23ed\\ufe0f  Skipped (already ingested): {filename}"

        document_id = insert_document(
            document_id=document_id,
            document_name=document_name,
            fingerprint=fingerprint,
            status="processing",
        )

        markdown = extract_text_from_pdf(pdf_path)
        if not markdown or len(markdown.strip()) < 500:
            raise ValueError("Extracted text too small or empty")

        chunks = chunk_markdown_document(markdown=markdown, document_name=document_name)
        if not chunks:
            raise ValueError("No chunks produced")

        embed_and_insert(chunks=chunks, document_id=document_id)
        update_document_status(document_id=document_id, status="done")

        return f"\\u2705 Success: {filename} ({len(chunks)} chunks)"

    except Exception as e:
        update_document_status(document_id=document_id, status="failed", error_message=str(e))
        return f"\\u274c Failed: {filename} — {type(e).__name__}: {e}\\n{traceback.format_exc()}"


def ingest_folder(pdf_folder: str):
    if not os.path.isdir(pdf_folder):
        raise ValueError(f"Not a directory: {pdf_folder}")

    pdf_files = [f for f in sorted(os.listdir(pdf_folder)) if f.lower().endswith(".pdf")]

    if not pdf_files:
        print("No PDF files found.")
        return

    print(f"Starting ingestion of {len(pdf_files)} PDFs (sequential)...\\n")

    for i, filename in enumerate(pdf_files, 1):
        pdf_path = os.path.join(pdf_folder, filename)
        print(f"[{i}/{len(pdf_files)}] {filename}")
        result = ingest_single(pdf_path, filename)
        print(result)
        print()

    print("Ingestion completed.")
'''

with open("/content/drive/MyDrive/rag_project/ingestion/ingest_folder.py", "w") as f:
    f.write(content)

print("✅ File written. Now run ingestion.")

✅ File written. Now run ingestion.


In [40]:
import importlib
import ingestion.ingest_folder
importlib.reload(ingestion.ingest_folder)

from ingestion.ingest_folder import ingest_folder
ingest_folder("/content/drive/MyDrive/rag_project/data")

Starting ingestion of 59 PDFs (sequential)...

[1/59] Acad-Calen-Revised-for-Winter-Semester-2022-23-for-Seniors-15-11-2022.pdf
❌ Failed: Acad-Calen-Revised-for-Winter-Semester-2022-23-for-Seniors-15-11-2022.pdf — OperationalError: SSL SYSCALL error: EOF detected

Traceback (most recent call last):
  File "/content/drive/MyDrive/rag_project/ingestion/ingest_folder.py", line 22, in ingest_single
    if document_exists_by_fingerprint(fingerprint):
       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/rag_project/ingestion/final_ingestion.py", line 158, in document_exists_by_fingerprint
    cursor.execute(
psycopg2.OperationalError: SSL SYSCALL error: EOF detected



[2/59] Acad-Calen-for-MBA-Freshers-2023-24.pdf





















































































































































Recognizing Text: 100%|██████████| 46/46 [00:02<00:00, 16.83it/s]


1 chunks are <100 chars
Ingested 3 chunks in 1 batches
✅ Success: Acad-Calen-for-MBA-Freshers-2023-24.pdf (3 chunks)

[3/59] Acad_Calendar_for_Fall_Semester_2025_26_Freshers.pdf


Recognizing Text: 100%|██████████| 51/51 [00:02<00:00, 20.25it/s]


1 chunks are <100 chars
Ingested 3 chunks in 1 batches
✅ Success: Acad_Calendar_for_Fall_Semester_2025_26_Freshers.pdf (3 chunks)

[4/59] Acad_Calendar_for_Fall_Semester_2025_26_Freshers_B_Tech_and_5year_pg_programmes.pdf


...

1 chunks are <100 chars
Ingested 3 chunks in 1 batches
✅ Success: accreditation.pdf (3 chunks)

[58/59] e-SANAD-Notification.pdf


Recognizing Text: 100%|██████████| 30/30 [00:15<00:00,  1.99it/s]
Detecting bboxes: 0it [00:00, ?it/s]


2 chunks are <100 chars
Ingested 5 chunks in 1 batches
✅ Success: e-SANAD-Notification.pdf (5 chunks)

[59/59] library-policy.pdf


Running OCR Error Detection: 100%|██████████| 1/1 [00:00<00:00, 79.41it/s]
Detecting bboxes: 0it [00:00, ?it/s]
Detecting bboxes: 0it [00:00, ?it/s]


3 chunks are <100 chars
Ingested 34 chunks in 1 batches
✅ Success: library-policy.pdf (34 chunks)

Ingestion completed.
